## Импорты и загрузка данных
Данные загружаются после интенсивной чистки в прошлых 2 ноутбуках. Я надеюсь, что это должно было избавить от почти всех проблем в написании имен и от значимого количества опечатка в фамилиях и отчествах

In [1]:
!pip install rapidfuzz sparse_dot_topn --quiet

import pandas as pd
import numpy as np
from rapidfuzz import fuzz, distance
from collections import defaultdict
from tqdm.notebook import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import awesome_cossim_topn

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [3]:
orcs = pd.read_parquet('/kaggle/input/aim-orcs-cleaning/orcs_clean_v2.parquet')
employees = pd.read_parquet('/kaggle/input/aim-orcs-cleaning/employees_clean_v2.parquet')

print(f"Loaded Orcs: {orcs.shape}")
print(f"Loaded Employees: {employees.shape}")

Loaded Orcs: (47633, 7)
Loaded Employees: (2011759, 7)


In [4]:
def make_concat(df):
    return (df['surname'] + ' ' + df['name'] + ' ' + df['fathername']).str.strip()

orcs['concat_fio'] = make_concat(orcs)
employees['concat_fio'] = make_concat(employees)

# Глобальные хранилища результатов
all_confident = pd.DataFrame()
all_review = pd.DataFrame()
solved_orcs = set()

print("Подготовка завершена.")

Подготовка завершена.


## Скоринг
5 полей
ФИО, Пол, ИНН, Паспорт, ДР

По этим полям мы пытаемся ответить на вопрос, является ли пара записей одинаковй.

Каждое поле может как-то проголосовать. При несовпадении оно голосует -1. При нейтральной ситуации голосуется 0. Примером нейтральной ситуации может быть наличие инн только у одного элемента пары. Если бы мы просто смотрели расстояния ливенштэйна - это было бы отличие в 12 символов. По большому счет нет разницы, отличается инн в 4 знаках или в 8 - это всё равно разный инн. Поэтому такая система с голосованиями ограничевает негативное влияние каждого фактора.

В случае совпадения или похожести каждое поле голосует с каким-то плюсом. Совпадение пола - это +1. Совпадение инн +3. Почти совпадение инн - это +2 или +1. 

Веса при голосовании за (насколько сильно каждый признак должен иметь занчение) перенастраивались после разглядывания "пограничных" значений. Например, только сопадения ФИО и пола должно быть недостаточно. 

In [ ]:
def get_name_score(s1, s2):
    return fuzz.token_sort_ratio(s1, s2)

def get_id_dist(s1, s2):
    return distance.DamerauLevenshtein.distance(s1, s2)

def get_smart_score(val_orc, val_emp, data_type):
    """
    Возвращает скор для пары значений

    data_type:
        'name' - ФИО (fuzzy)
        'gender' - пол (exact)
        'strong_id' - ИНН, паспорт (сильный сигнал)
        'date' - дата рождения (слабее чем id)
    """
    # Если данных нет -> 0 баллов (Нейтрально)
    if not val_orc or not val_emp:
        return 0

    # 1. ИМЕНА (fuzzy matching)
    if data_type == 'name':
        sim = get_name_score(val_orc, val_emp)
        if sim >= 95: return 3   # Почти идеал
        if sim >= 85: return 2   # Очень похоже
        if sim >= 65: return 1   # Похоже
        if sim >= 50: return 0   # Сомнительно
        return -1                # Не похоже

    # 2. ПОЛ (exact)
    if data_type == 'gender':
        if val_orc not in ['м', 'ж'] or val_emp not in ['м', 'ж']:
            return 0
        return 1 if val_orc == val_emp else -1

    # 3. СИЛЬНЫЕ ID (ИНН, Паспорт) - повышенные баллы
    if data_type == 'strong_id':
        if abs(len(val_orc) - len(val_emp)) > 2:
            return -1

        dist = get_id_dist(val_orc, val_emp)
        if dist == 0: return 3   # Идеал (+3) — было +2
        if dist == 1: return 2   # 1 ошибка (+2) — было +1
        if dist == 2: return 1   # 2 ошибки (+1) — было 0
        return -1                # 3+ ошибки (-1)

    # 4. ДАТА РОЖДЕНИЯ (слабее чем id, много коллизий)
    if data_type == 'date':
        if abs(len(val_orc) - len(val_emp)) > 2:
            return -1

        dist = get_id_dist(val_orc, val_emp)
        if dist == 0: return 2   # Идеал (+2)
        if dist == 1: return 1   # 1 ошибка (+1)
        if dist == 2: return 0   # 2 ошибки (0)
        return -1                # 3+ ошибки (-1)

    return 0


def rate_candidates(pairs_df, orcs_df, emps_df):
    df = pairs_df.copy()

    cols_id = ['passport', 'inn', 'birthdate']

    orc_indices = df['orc_idx'].values
    emp_indices = df['emp_idx'].values

    df['fio_orc'] = orcs_df.loc[orc_indices, 'concat_fio'].values
    df['fio_emp'] = emps_df.loc[emp_indices, 'concat_fio'].values

    df['gender_orc'] = orcs_df.loc[orc_indices, 'gender'].values
    df['gender_emp'] = emps_df.loc[emp_indices, 'gender'].values

    # ID поля
    for col in cols_id:
        if col in orcs_df.columns:
            df[f'{col}_orc'] = orcs_df.loc[orc_indices, col].values
        else:
            df[f'{col}_orc'] = ""

        if col in emps_df.columns:
            df[f'{col}_emp'] = emps_df.loc[emp_indices, col].values
        else:
            df[f'{col}_emp'] = ""

    # === СКОРИНГ ===

    # ФИО
    df['score_fio'] = [get_smart_score(x, y, 'name') for x, y in zip(df['fio_orc'], df['fio_emp'])]

    # Пол
    df['score_gender'] = [get_smart_score(x, y, 'gender') for x, y in zip(df['gender_orc'], df['gender_emp'])]

    # Паспорт и ИНН — СИЛЬНЫЕ ID (повышенные баллы)
    df['score_pass'] = [get_smart_score(x, y, 'strong_id') for x, y in zip(df['passport_orc'], df['passport_emp'])]
    df['score_inn'] = [get_smart_score(x, y, 'strong_id') for x, y in zip(df['inn_orc'], df['inn_emp'])]

    # Дата рождения — обычный id (много коллизий)
    df['score_date'] = [get_smart_score(x, y, 'date') for x, y in zip(df['birthdate_orc'], df['birthdate_emp'])]

    # === ИТОГИ ===
    score_cols = ['score_fio', 'score_pass', 'score_inn', 'score_gender', 'score_date']
    df['total_score'] = df[score_cols].sum(axis=1)

    neg_counts = (df[score_cols] == -1).sum(axis=1)

    # Критерии уверенности:
    df['is_confident'] = (
        ((neg_counts == 0) & (df['total_score'] >= 4)) |
        ((neg_counts <= 1) & (df['total_score'] >= 5))
    )

    return df

## Хэши (чтобы быстро находить пары на расстоянии 1 или 2)

In [ ]:
def find_candidates_even_odd(active_orcs, employees, col_name):
    # Берем значения
    orc_s = active_orcs[col_name].fillna('')
    emp_s = employees[col_name].fillna('')

    # Отсекаем короткие (<4)
    orc_s = orc_s[orc_s.str.len() >= 4]
    emp_s = emp_s[emp_s.str.len() >= 4]

    # Индекс Сотрудников
    even_map = defaultdict(list)
    odd_map = defaultdict(list)

    for idx, val in emp_s.items():
        even_map[val[::2]].append(idx)
        odd_map[val[1::2]].append(idx)

    candidates = []

    for idx, val in tqdm(orc_s.items(), desc=f"Fuzzy {col_name} (Subst)", leave=False):
        potentials = set()
        # Если ошибка в нечетном, четный хэш совпадет
        potentials.update(even_map.get(val[::2], []))
        # Если ошибка в четном, нечетный хэш совпадет
        potentials.update(odd_map.get(val[1::2], []))

        for emp_idx in potentials:
            # Проверка
            if get_id_dist(val, emp_s[emp_idx]) == 1:
                candidates.append({'orc_idx': idx, 'emp_idx': emp_idx})

    return pd.DataFrame(candidates)

# 2. Поиск удаления/вставки (Distance=1, Deletion)
def find_candidates_deletion(active_orcs, employees, col_name):
    orc_s = active_orcs[col_name].fillna('')
    emp_s = employees[col_name].fillna('')

    # Индекс удалений (Сотрудники)
    # Ключ: строка_без_1_символа -> [id]
    del_index = defaultdict(list)

    for idx, val in emp_s.items():
        if len(val) < 4: continue
        # Оригинал
        del_index[val].append(idx)
        # Удаления
        for i in range(len(val)):
            sub = val[:i] + val[i+1:]
            del_index[sub].append(idx)

    candidates = []

    for idx, val in tqdm(orc_s.items(), desc=f"Fuzzy {col_name} (Del)", leave=False):
        if len(val) < 4: continue
        potentials = set()

        # Генерируем варианты орка
        variants = {val} # Сам орк
        for i in range(len(val)):
            variants.add(val[:i] + val[i+1:])

        for v in variants:
            potentials.update(del_index.get(v, []))

        for emp_idx in potentials:
            if get_id_dist(val, emp_s[emp_idx]) == 1:
                candidates.append({'orc_idx': idx, 'emp_idx': emp_idx})

    return pd.DataFrame(candidates)

# 3. Поиск Distance=2 (через 3 части)
# Если разбить строку на 3 части, то при 2 ошибках ХОТЯ БЫ ОДНА часть останется нетронутой.
# Это принцип Pigeonhole Principle.
def find_candidates_dist_2(active_orcs, employees, col_name):
    orc_s = active_orcs[col_name].fillna('')
    emp_s = employees[col_name].fillna('')

    # Только длинные (>6), иначе частей не наберешь
    orc_s = orc_s[orc_s.str.len() >= 6]
    emp_s = emp_s[emp_s.str.len() >= 6]

    # Индексы частей
    # part0: первая треть, part1: середина, part2: конец
    index_p0 = defaultdict(list)
    index_p1 = defaultdict(list)
    index_p2 = defaultdict(list)

    def get_parts(s):
        n = len(s)
        k = n // 3
        return s[:k], s[k:2*k], s[2*k:]

    for idx, val in emp_s.items():
        p0, p1, p2 = get_parts(val)
        index_p0[p0].append(idx)
        index_p1[p1].append(idx)
        index_p2[p2].append(idx)

    candidates = []

    for idx, val in tqdm(orc_s.items(), desc=f"Fuzzy {col_name} (Dist=2)", leave=False):
        p0, p1, p2 = get_parts(val)
        potentials = set()

        potentials.update(index_p0.get(p0, []))
        potentials.update(index_p1.get(p1, []))
        potentials.update(index_p2.get(p2, []))

        for emp_idx in potentials:
            # Тут dist мб до 2
            if get_id_dist(val, emp_s[emp_idx]) <= 2:
                candidates.append({'orc_idx': idx, 'emp_idx': emp_idx})

    return pd.DataFrame(candidates)

In [ ]:
def find_candidates_deletion_blocked(active_orcs, employees, col_name, block_len=2):
    """Поиск Dist=1 с блокировкой по первым N символам"""

    # Копируем и фильтруем пустые/короткие СРАЗУ
    orc_series = active_orcs[col_name].fillna('').copy()
    emp_series = employees[col_name].fillna('').copy()

    # Жёсткая фильтрация
    orc_mask = orc_series.str.len() >= max(4, block_len + 1)
    emp_mask = emp_series.str.len() >= max(4, block_len + 1)

    orc_s = orc_series[orc_mask]
    emp_s = emp_series[emp_mask]

    if len(orc_s) == 0 or len(emp_s) == 0:
        return pd.DataFrame(columns=['orc_idx', 'emp_idx'])

    # Группируем сотрудников по блокам (первые N символов)
    emp_by_block = defaultdict(dict)
    for idx, val in emp_s.items():
        if len(val) < block_len:
            continue
        block = val[:block_len]
        emp_by_block[block][idx] = val

    # Индекс удалений внутри блоков
    del_index_by_block = {}

    for block, emp_dict in emp_by_block.items():
        del_index = defaultdict(list)
        for idx, val in emp_dict.items():
            del_index[val].append(idx)
            for i in range(len(val)):
                sub = val[:i] + val[i+1:]
                del_index[sub].append(idx)
        del_index_by_block[block] = (del_index, emp_dict)

    candidates = []

    for idx, val in tqdm(orc_s.items(), desc=f"Fuzzy {col_name} Dist=1", leave=False):
        if len(val) < block_len:
            continue

        block = val[:block_len]

        if block not in del_index_by_block:
            continue

        del_index, emp_dict = del_index_by_block[block]
        potentials = set()

        # Генерируем варианты орка (сам + все удаления)
        variants = {val}
        for i in range(len(val)):
            variants.add(val[:i] + val[i+1:])

        for v in variants:
            potentials.update(del_index.get(v, []))

        for emp_idx in potentials:
            emp_val = emp_dict.get(emp_idx, '')
            if emp_val and get_id_dist(val, emp_val) == 1:
                candidates.append({'orc_idx': idx, 'emp_idx': emp_idx})

    return pd.DataFrame(candidates) if candidates else pd.DataFrame(columns=['orc_idx', 'emp_idx'])


def find_candidates_dist_2_blocked(active_orcs, employees, col_name, block_len=2):
    """Поиск Dist<=2 с блокировкой по первым N символам"""

    # Копируем и фильтруем
    orc_series = active_orcs[col_name].fillna('').copy()
    emp_series = employees[col_name].fillna('').copy()

    # Для разбиения на 3 части нужно минимум 6 символов
    min_len = max(6, block_len + 1)

    orc_mask = orc_series.str.len() >= min_len
    emp_mask = emp_series.str.len() >= min_len

    orc_s = orc_series[orc_mask]
    emp_s = emp_series[emp_mask]

    if len(orc_s) == 0 or len(emp_s) == 0:
        return pd.DataFrame(columns=['orc_idx', 'emp_idx'])

    def get_parts(s):
        n = len(s)
        k = n // 3
        return s[:k], s[k:2*k], s[2*k:]

    # Группируем сотрудников по блокам
    emp_by_block = defaultdict(list)
    for idx, val in emp_s.items():
        if len(val) >= min_len:
            block = val[:block_len]
            emp_by_block[block].append((idx, val))

    # Строим индексы частей для каждого блока
    index_by_block = {}
    for block, emp_list in emp_by_block.items():
        index_p0 = defaultdict(list)
        index_p1 = defaultdict(list)
        index_p2 = defaultdict(list)
        emp_vals = {}

        for idx, val in emp_list:
            p0, p1, p2 = get_parts(val)
            index_p0[p0].append(idx)
            index_p1[p1].append(idx)
            index_p2[p2].append(idx)
            emp_vals[idx] = val

        index_by_block[block] = (index_p0, index_p1, index_p2, emp_vals)

    candidates = []

    for idx, val in tqdm(orc_s.items(), desc=f"Fuzzy {col_name} Dist=2", leave=False):
        if len(val) < min_len:
            continue

        block = val[:block_len]

        if block not in index_by_block:
            continue

        index_p0, index_p1, index_p2, emp_vals = index_by_block[block]

        p0, p1, p2 = get_parts(val)
        potentials = set()

        potentials.update(index_p0.get(p0, []))
        potentials.update(index_p1.get(p1, []))
        potentials.update(index_p2.get(p2, []))

        for emp_idx in potentials:
            emp_val = emp_vals.get(emp_idx, '')
            if emp_val and get_id_dist(val, emp_val) <= 2:
                candidates.append({'orc_idx': idx, 'emp_idx': emp_idx})

    return pd.DataFrame(candidates) if candidates else pd.DataFrame(columns=['orc_idx', 'emp_idx'])

## Основаная логика: находим пары и проверяем по критерию

На каждом этапе смотрим только тех орков, которых ещё не отобрали. Идея была в том, чтобы более тяжелые поиски делать на меньшем числе орков.
Результаты складываем в 2 кучки - тех, в ком уверены и пограничные случаи на ручное ревью

#### ИНН - точное совпадение, или на расстоянии 1-2 замены/удаления

In [ ]:
active_mask = ~orcs.index.isin(solved_orcs)
active_orcs = orcs[active_mask].copy()

print(f"--- ЭТАП 1: ИНН (Вход: {len(active_orcs)}) ---")

if len(active_orcs) > 0:
    # 1. Exact Match
    print("Поиск Exact...")
    # Игнорим пустые строки
    valid_orc = active_orcs[active_orcs['inn'].str.len() > 0]
    valid_emp = employees[employees['inn'].str.len() > 0]

    cand_exact = valid_orc[['inn']].reset_index().merge(
        valid_emp[['inn']].reset_index(),
        on='inn',
        suffixes=('_orc', '_emp')
    ).rename(columns={'index_orc': 'orc_idx', 'index_emp': 'emp_idx'})
    print(f"  Exact: {len(cand_exact)}")

    # Определяем оставшихся для Fuzzy
    remaining_orcs = active_orcs[~active_orcs.index.isin(cand_exact['orc_idx'])]

    # 2. Fuzzy Match (Dist=1) - Deletion Join
    cand_fuzzy_1 = pd.DataFrame()
    if not remaining_orcs.empty:
        print("Поиск Fuzzy (Dist=1)...")
        cand_fuzzy_1 = find_candidates_deletion(remaining_orcs, employees, 'inn')
        print(f"  Fuzzy 1: {len(cand_fuzzy_1)}")

        # Обновляем оставшихся (чтобы не искать dist=2 для тех, кого нашли с dist=1)
        remaining_orcs = remaining_orcs[~remaining_orcs.index.isin(cand_fuzzy_1['orc_idx'])]

    # 3. Fuzzy Match (Dist=2) - 3-gram Blocking
    cand_fuzzy_2 = pd.DataFrame()
    if not remaining_orcs.empty:
        print("Поиск Fuzzy (Dist=2)...")
        cand_fuzzy_2 = find_candidates_dist_2(remaining_orcs, employees, 'inn')
        print(f"  Fuzzy 2: {len(cand_fuzzy_2)}")

    # Объединяем всех кандидатов
    all_cand = pd.concat([cand_exact, cand_fuzzy_1, cand_fuzzy_2]).drop_duplicates(subset=['orc_idx', 'emp_idx'])

    if not all_cand.empty:
        scored = rate_candidates(all_cand, orcs, employees)

        # --- ЛОГИКА РАЗДЕЛЕНИЯ ---

        # 1. CONFIDENT (Уверенные)
        # Критерий: is_confident (из функции rate_candidates)
        mask_conf = scored['is_confident'] == True
        conf = scored[mask_conf].copy()
        conf = conf.sort_values('total_score', ascending=False).drop_duplicates('orc_idx')

        # 2. REVIEW (Threshold Score >= 4)
        mask_review = (~mask_conf) & (scored['total_score'] >= 4)
        review = scored[mask_review].copy()

        # СОХРАНЕНИЕ
        all_confident = pd.concat([all_confident, conf])
        solved_orcs.update(conf['orc_idx'].unique())

        if not review.empty:
            all_review = pd.concat([all_review, review])

        print(f"✅ Решено по ИНН: {len(conf)}")
        print(f"⚠️ Добавлено в кандидаты (Review): {len(review)}")

print(f"Осталось: {len(orcs) - len(solved_orcs)}")

--- ЭТАП 1: ИНН (Вход: 47633) ---
Поиск Exact...
  Exact: 4042
Поиск Fuzzy (Dist=1)...


Fuzzy inn (Del): 0it [00:00, ?it/s]

  Fuzzy 1: 341
Поиск Fuzzy (Dist=2)...


Fuzzy inn (Dist=2): 0it [00:00, ?it/s]

  Fuzzy 2: 319
✅ Решено по ИНН: 4493
⚠️ Добавлено в кандидаты (Review): 2
Осталось: 43140


#### Пасспорт - точное совпадение, или на расстоянии 1-2 замены/удаления

Пасспорта на расстоянии 2 считались минут 15, поэтому добавил блокировки, стало 2 минуты

In [ ]:
active_mask = ~orcs.index.isin(solved_orcs)
active_orcs = orcs[active_mask].copy()

print(f"\n--- ЭТАП 2: ПАСПОРТ (Вход: {len(active_orcs)}) ---")

col_pass = 'passport'

if len(active_orcs) > 0:
    # 1. Exact Match
    print("Поиск Exact...")
    valid_orc = active_orcs[active_orcs[col_pass].str.len() > 0]
    valid_emp = employees[employees[col_pass].str.len() > 0]

    cand_exact = valid_orc[[col_pass]].reset_index().merge(
        valid_emp[[col_pass]].reset_index(),
        on=col_pass,
        suffixes=('_orc', '_emp')
    ).rename(columns={'index_orc': 'orc_idx', 'index_emp': 'emp_idx'})
    print(f"  Exact: {len(cand_exact)}")

    remaining_orcs = active_orcs[~active_orcs.index.isin(cand_exact['orc_idx'])]

    # 2. Fuzzy Match (Dist=1) - С БЛОКИРОВКОЙ
    cand_fuzzy_1 = pd.DataFrame()
    if not remaining_orcs.empty:
        print("Поиск Fuzzy (Dist=1) с блокировкой...")
        cand_fuzzy_1 = find_candidates_deletion_blocked(remaining_orcs, employees, col_pass, block_len=2)
        print(f"  Fuzzy 1: {len(cand_fuzzy_1)}")

        remaining_orcs = remaining_orcs[~remaining_orcs.index.isin(cand_fuzzy_1['orc_idx'])]

    # 3. Fuzzy Match (Dist=2) - С БЛОКИРОВКОЙ
    cand_fuzzy_2 = pd.DataFrame()
    if not remaining_orcs.empty:
        print("Поиск Fuzzy (Dist=2) с блокировкой...")
        cand_fuzzy_2 = find_candidates_dist_2_blocked(remaining_orcs, employees, col_pass, block_len=2)
        print(f"  Fuzzy 2: {len(cand_fuzzy_2)}")

    all_cand = pd.concat([cand_exact, cand_fuzzy_1, cand_fuzzy_2]).drop_duplicates(subset=['orc_idx', 'emp_idx'])

    if not all_cand.empty:
        scored = rate_candidates(all_cand, orcs, employees)

        mask_conf = scored['is_confident'] == True
        conf = scored[mask_conf].copy()
        conf = conf.sort_values('total_score', ascending=False).drop_duplicates('orc_idx')

        mask_review = (~mask_conf) & (scored['total_score'] >= 4)
        review = scored[mask_review].copy()

        all_confident = pd.concat([all_confident, conf])
        solved_orcs.update(conf['orc_idx'].unique())

        if not review.empty:
            all_review = pd.concat([all_review, review])

        print("-" * 30)
        print(f"✅ Решено по Паспорту: {len(conf)}")
        print(f"⚠️ Review: {len(review)}")

print(f"Осталось нерешённых: {len(orcs) - len(solved_orcs)}")


--- ЭТАП 2: ПАСПОРТ (Вход: 43140) ---
Поиск Exact...
  Exact: 6047
Поиск Fuzzy (Dist=1) с блокировкой...


Fuzzy passport Dist=1: 0it [00:00, ?it/s]

  Fuzzy 1: 824
Поиск Fuzzy (Dist=2) с блокировкой...


Fuzzy passport Dist=2: 0it [00:00, ?it/s]

  Fuzzy 2: 15972
------------------------------
✅ Решено по Паспорту: 6474
⚠️ Review: 17
Осталось нерешённых: 36666


#### ДР - точное совпадение, дало 4млн пар-кандидатов

In [ ]:
active_mask = ~orcs.index.isin(solved_orcs)
active_orcs = orcs[active_mask].copy()

print(f"\n--- ЭТАП 3: ДАТА РОЖДЕНИЯ (Вход: {len(active_orcs)}) ---")

col_date = 'birthdate'

if col_date in active_orcs.columns and col_date in employees.columns:

    # 1. ПОДГОТОВКА (ФИЛЬТРАЦИЯ ПУСТЫХ)

    orcs_with_date = active_orcs[active_orcs[col_date].str.len() > 5].copy()
    emps_with_date = employees[employees[col_date].str.len() > 5].copy()

    print(f"Орков с датой: {len(orcs_with_date)}")
    print(f"Сотрудников с датой: {len(emps_with_date)}")

    # 2. ПОИСК (MERGE)
    print("Merge по точной Дате...")
    cand_dob = orcs_with_date[[col_date]].reset_index().merge(
        emps_with_date[[col_date]].reset_index(),
        on=col_date,
        suffixes=('_orc', '_emp')
    ).rename(columns={'index_orc': 'orc_idx', 'index_emp': 'emp_idx'})

    print(f"Найдено кандидатов: {len(cand_dob)}")

    if not cand_dob.empty:
        # 3. ОЦЕНКА
        print("Расчет скоров...")
        scored_dob = rate_candidates(cand_dob, orcs, employees)

        # 4. ФИЛЬТРАЦИЯ

        # CONFIDENT:
        mask_conf = scored_dob['is_confident'] == True

        # REVIEW: (Не уверен, но Score >= 4)
        mask_rev = (~mask_conf) & (scored_dob['total_score'] >= 4)

        conf_dob = scored_dob[mask_conf].copy()
        rev_dob = scored_dob[mask_rev].copy()

        # 5. СОХРАНЕНИЕ
        if not conf_dob.empty:
            # Дедупликация (берем лучшего)
            conf_dob = conf_dob.sort_values('total_score', ascending=False).drop_duplicates('orc_idx')

            all_confident = pd.concat([all_confident, conf_dob])
            solved_orcs.update(conf_dob['orc_idx'].unique())

        if not rev_dob.empty:
            all_review = pd.concat([all_review, rev_dob])

        print("-" * 30)
        print(f"✅ Дата Уверены: {len(conf_dob)}")
        print(f"⚠️ Дата Ревью:   {len(rev_dob)}")
        print(f"🗑️ Отброшено:    {len(scored_dob) - len(conf_dob) - len(rev_dob)}")

    else:
        print("Совпадений по датам не найдено.")

else:
    print(f"Колонка '{col_date}' отсутствует в данных.")

print(f"Осталось нерешенных: {len(orcs) - len(solved_orcs)}")


--- ЭТАП 3: ДАТА РОЖДЕНИЯ (Вход: 36666) ---
Орков с датой: 33309
Сотрудников с датой: 1871037
Merge по точной Дате...
Найдено кандидатов: 4095645
Расчет скоров...
------------------------------
✅ Дата Уверены: 2184
⚠️ Дата Ревью:   62
🗑️ Отброшено:    4093399
Осталось нерешенных: 34482


#### Точные совпадения фамилии или имени+отчества
С учетом того, что данные я уже неплохо почистил от опечаток, я не делал тут fuzzy search. Подозреваю, что это бы очень долго.

In [ ]:
active_mask = ~orcs.index.isin(solved_orcs)
active_orcs = orcs[active_mask].copy()

print(f"\n--- ЭТАП 4: ФАМИЛИЯ (Вход: {len(active_orcs)}) ---")


active_orcs = active_orcs[active_orcs['surname'] != ""]
active_orcs = active_orcs[active_orcs['surname'].str.len() >= 2]

active_orcs['key_sur'] = active_orcs['surname']
employees['key_sur'] = employees['surname']


# Исключаем слишком короткие (1 буква), чтобы не взорвать память, остальное берем
active_orcs = active_orcs[active_orcs['key_sur'].str.len() >= 2]

# 2. Поиск (Merge по Фамилии)
print("Merge по точному совпадению Фамилии...")
cand_sur = active_orcs[['key_sur']].reset_index().merge(
    employees[['key_sur']].reset_index(),
    on='key_sur',
    suffixes=('_orc', '_emp')
).rename(columns={'index_orc': 'orc_idx', 'index_emp': 'emp_idx'})

print(f"Найдено кандидатов: {len(cand_sur)}")

if not cand_sur.empty:
    # 3. Оценка
    print("Расчет скоров...")
    scored_sur = rate_candidates(cand_sur, orcs, employees)


    mask_conf = scored_sur['is_confident'] == True


    mask_rev = (~mask_conf) & (scored_sur['total_score'] >= 4)

    conf_sur = scored_sur[mask_conf].copy()
    rev_sur = scored_sur[mask_rev].copy()

    # 5. Сохранение
    if not conf_sur.empty:
        # Дедупликация (берем кандидата с лучшим скором)
        conf_sur = conf_sur.sort_values('total_score', ascending=False).drop_duplicates('orc_idx')

        all_confident = pd.concat([all_confident, conf_sur])
        solved_orcs.update(conf_sur['orc_idx'].unique())

    if not rev_sur.empty:
        all_review = pd.concat([all_review, rev_sur])

    print("-" * 30)
    print(f"✅ Фамилия Уверены: {len(conf_sur)}")
    print(f"⚠️ Фамилия Ревью (Score >= 3): {len(rev_sur)}")
    print(f"🗑️ Отброшено (Однофамильцы): {len(scored_sur) - len(conf_sur) - len(rev_sur)}")

print(f"Осталось нерешенных: {len(orcs) - len(solved_orcs)}")


--- ЭТАП 4: ФАМИЛИЯ (Вход: 34482) ---
Merge по точному совпадению Фамилии...
Найдено кандидатов: 1682589
Расчет скоров...
------------------------------
✅ Фамилия Уверены: 129
⚠️ Фамилия Ревью (Score >= 3): 7
🗑️ Отброшено (Однофамильцы): 1682453
Осталось нерешенных: 34353


In [ ]:
active_mask = ~orcs.index.isin(solved_orcs)
active_orcs = orcs[active_mask].copy()

print(f"\n--- ЭТАП 5: ИМЯ + ОТЧЕСТВО (Вход: {len(active_orcs)}) ---")

# 1. Подготовка ключей
def make_nf_key(df):
    # fillna('') обязателен, т.к. при сложении со строкой NaN дает NaN
    return (df['name'].fillna('') + ' ' + df['fathername'].fillna('')).str.lower().str.strip()

active_orcs['key_nf'] = make_nf_key(active_orcs)
employees['key_nf'] = make_nf_key(employees)

# Фильтруем пустые/короткие (меньше 3 символов - это мусор типа "я я")
active_orcs = active_orcs[active_orcs['key_nf'].str.len() >= 3]

# 2. Поиск (Merge)
print("Merge по точному Имя+Отчество...")
cand_nf = active_orcs[['key_nf']].reset_index().merge(
    employees[['key_nf']].reset_index(),
    on='key_nf',
    suffixes=('_orc', '_emp')
).rename(columns={'index_orc': 'orc_idx', 'index_emp': 'emp_idx'})

print(f"Найдено кандидатов: {len(cand_nf)}")

if not cand_nf.empty:
    # 3. Оценка
    print("Расчет скоров...")
    scored_nf = rate_candidates(cand_nf, orcs, employees)

    # 4. Фильтрация
    # CONFIDENT: is_confident (Нет конфликтов + Высокий балл)
    mask_conf = scored_nf['is_confident'] == True

    # REVIEW: Score >= 4
    mask_rev = (~mask_conf) & (scored_nf['total_score'] >= 4)

    conf_nf = scored_nf[mask_conf].copy()
    rev_nf = scored_nf[mask_rev].copy()

    # 5. Сохранение
    if not conf_nf.empty:
        conf_nf = conf_nf.sort_values('total_score', ascending=False).drop_duplicates('orc_idx')
        all_confident = pd.concat([all_confident, conf_nf])
        solved_orcs.update(conf_nf['orc_idx'].unique())

    if not rev_nf.empty:
        all_review = pd.concat([all_review, rev_nf])

    print("-" * 30)
    print(f"✅ Имя+Отч Уверены: {len(conf_nf)}")
    print(f"⚠️ Имя+Отч Ревью (Score >= 3): {len(rev_nf)}")
    print(f"🗑️ Отброшено: {len(scored_nf) - len(conf_nf) - len(rev_nf)}")

print(f"Осталось нерешенных: {len(orcs) - len(solved_orcs)}")


--- ЭТАП 5: ИМЯ + ОТЧЕСТВО (Вход: 34353) ---
Merge по точному Имя+Отчество...
Найдено кандидатов: 338874
Расчет скоров...
------------------------------
✅ Имя+Отч Уверены: 101
⚠️ Имя+Отч Ревью (Score >= 3): 14
🗑️ Отброшено: 338759
Осталось нерешенных: 34252


#### Попытка найти хороших кандидатов по общей похожести фио
В итоге не нашло почти ничего сверх найденного выше

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import awesome_cossim_topn
import scipy.sparse as sp

print("\n=== ЭТАП 6: TF-IDF FUZZY ФИО (ДВОЙНАЯ БЛОКИРОВКА) ===")

active_mask = ~orcs.index.isin(solved_orcs)
active_orcs_fio = orcs[active_mask].copy()

print(f"Нерешённых орков: {len(active_orcs_fio)}")

# Фильтруем пустые ФИО
active_orcs_fio = active_orcs_fio[active_orcs_fio['concat_fio'].fillna('').str.len() >= 5]
valid_employees = employees[employees['concat_fio'].fillna('').str.len() >= 5].copy()

# Создаём ключ блокировки: пол + первая буква фамилии
def make_block_key(df):
    gender = df['gender'].fillna('x').astype(str)
    surname_first = df['surname'].fillna('').str[:1].str.lower()
    # Заменяем пустые на 'x'
    surname_first = surname_first.replace('', 'x')
    return gender + '_' + surname_first

active_orcs_fio['block_key'] = make_block_key(active_orcs_fio)
valid_employees['block_key'] = make_block_key(valid_employees)

# Смотрим распределение блоков
print(f"\nОрков с ФИО >= 5: {len(active_orcs_fio)}")
print(f"Сотрудников с ФИО >= 5: {len(valid_employees)}")
print(f"Уникальных блоков у орков: {active_orcs_fio['block_key'].nunique()}")
print(f"Уникальных блоков у сотрудников: {valid_employees['block_key'].nunique()}")

# Группируем сотрудников по блокам заранее
emp_groups = {k: v for k, v in valid_employees.groupby('block_key')}

print(f"\nПримеры размеров блоков сотрудников:")
sizes = [(k, len(v)) for k, v in emp_groups.items()]
sizes.sort(key=lambda x: -x[1])
for k, s in sizes[:10]:
    print(f"  {k}: {s}")


def run_tfidf_for_block(orcs_block, emps_block, threshold=0.55, topn=5):
    """TF-IDF matching для одного блока"""

    if len(orcs_block) == 0 or len(emps_block) == 0:
        return pd.DataFrame(columns=['orc_idx', 'emp_idx', 'tfidf_sim'])

    orcs_text = orcs_block['concat_fio'].fillna('').values
    emps_text = emps_block['concat_fio'].fillna('').values

    orcs_idx = orcs_block.index.values
    emps_idx = emps_block.index.values

    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 3),
        min_df=1,
        max_df=0.95,
        dtype=np.float32
    )

    all_text = np.concatenate([orcs_text, emps_text])
    vectorizer.fit(all_text)

    orcs_tfidf = vectorizer.transform(orcs_text)
    emps_tfidf = vectorizer.transform(emps_text)

    matches = awesome_cossim_topn(
        orcs_tfidf,
        emps_tfidf.T,
        ntop=topn,
        lower_bound=threshold,
        use_threads=True,
        n_jobs=4
    )

    coo = matches.tocoo()

    if len(coo.data) == 0:
        return pd.DataFrame(columns=['orc_idx', 'emp_idx', 'tfidf_sim'])

    return pd.DataFrame({
        'orc_idx': orcs_idx[coo.row],
        'emp_idx': emps_idx[coo.col],
        'tfidf_sim': coo.data
    })


# Запуск по блокам
all_tfidf_candidates = []
orc_groups = active_orcs_fio.groupby('block_key')

total_blocks = len(orc_groups)
processed = 0

for block_key, orcs_block in tqdm(orc_groups, desc="TF-IDF блоки", total=total_blocks):

    # Получаем соответствующий блок сотрудников
    if block_key not in emp_groups:
        continue

    emps_block = emp_groups[block_key]

    # Пропускаем слишком большие блоки (защита)
    if len(emps_block) > 200000:
        print(f"  ⚠️ Пропуск блока {block_key}: {len(emps_block)} сотрудников (слишком много)")
        continue

    block_candidates = run_tfidf_for_block(
        orcs_block,
        emps_block,
        threshold=0.55,
        topn=5
    )

    if not block_candidates.empty:
        all_tfidf_candidates.append(block_candidates)

    processed += 1

print(f"\nОбработано блоков: {processed}/{total_blocks}")

# Объединяем результаты
if all_tfidf_candidates:
    cand_tfidf = pd.concat(all_tfidf_candidates, ignore_index=True)
    cand_tfidf = cand_tfidf.drop_duplicates(subset=['orc_idx', 'emp_idx'])

    print(f"Всего кандидатов TF-IDF: {len(cand_tfidf)}")

    # Скоринг
    print("Расчёт скоров...")
    scored_tfidf = rate_candidates(cand_tfidf[['orc_idx', 'emp_idx']], orcs, employees)

    scored_tfidf = scored_tfidf.merge(
        cand_tfidf[['orc_idx', 'emp_idx', 'tfidf_sim']],
        on=['orc_idx', 'emp_idx'],
        how='left'
    )

    # Критерии принятия
    score_cols = ['score_fio', 'score_pass', 'score_inn', 'score_gender', 'score_date']
    neg_counts = (scored_tfidf[score_cols] == -1).sum(axis=1)

    mask_accept = (
        scored_tfidf['is_confident'] == True
    )

    conf_tfidf = scored_tfidf[mask_accept].copy()
    conf_tfidf = conf_tfidf.sort_values(
        ['total_score', 'tfidf_sim'],
        ascending=[False, False]
    ).drop_duplicates('orc_idx')

    print("-" * 30)
    print(f"✅ Принято по TF-IDF ФИО: {len(conf_tfidf)}")

    all_confident = pd.concat([all_confident, conf_tfidf])
    solved_orcs.update(conf_tfidf['orc_idx'].unique())

    # Review
    mask_review = (~mask_accept) & (scored_tfidf['total_score'] >= 4)
    review_tfidf = scored_tfidf[mask_review]

    if not review_tfidf.empty:
        all_review = pd.concat([all_review, review_tfidf])
        print(f"⚠️ Review: {len(review_tfidf)}")

else:
    print("TF-IDF не нашёл кандидатов")

print(f"\nОсталось нерешённых: {len(orcs) - len(solved_orcs)}")


=== ЭТАП 6: TF-IDF FUZZY ФИО (ДВОЙНАЯ БЛОКИРОВКА) ===
Нерешённых орков: 34252

Орков с ФИО >= 5: 34252
Сотрудников с ФИО >= 5: 2011759
Уникальных блоков у орков: 65
Уникальных блоков у сотрудников: 70

Примеры размеров блоков сотрудников:
  ж_к: 138517
  м_к: 127643
  ж_с: 95396
  м_с: 87764
  ж_б: 78273
  ж_м: 76034
  м_б: 75013
  м_м: 72470
  ж_п: 67693
  м_п: 64764


TF-IDF блоки:   0%|          | 0/65 [00:00<?, ?it/s]

/tmp/ipykernel_283/3152298782.py:69: DeprecationWarning: `awesome_cossim_topn` function will be removed and (partially) replaced with `sp_matmul_topn`. See the migration guide at 'https://github.com/ing-bank/sparse_dot_topn#readme'.
  matches = awesome_cossim_topn(
/tmp/ipykernel_283/3152298782.py:69: DeprecationWarning: `awesome_cossim_topn` function will be removed and (partially) replaced with `sp_matmul_topn`. See the migration guide at 'https://github.com/ing-bank/sparse_dot_topn#readme'.
  matches = awesome_cossim_topn(
/tmp/ipykernel_283/3152298782.py:69: DeprecationWarning: `awesome_cossim_topn` function will be removed and (partially) replaced with `sp_matmul_topn`. See the migration guide at 'https://github.com/ing-bank/sparse_dot_topn#readme'.
  matches = awesome_cossim_topn(
/tmp/ipykernel_283/3152298782.py:69: DeprecationWarning: `awesome_cossim_topn` function will be removed and (partially) replaced with `sp_matmul_topn`. See the migration guide at 'https://github.com/ing


Обработано блоков: 65/65
Всего кандидатов TF-IDF: 3308
Расчёт скоров...
------------------------------
✅ Принято по TF-IDF ФИО: 9
⚠️ Review: 20

Осталось нерешённых: 34243


## Подготовка сабмита
Последние 2 сабмита были all_confident и all_confident + all_review. Второй оказался лучше

- all_confident - скор 0.9355
- all_confident + all_review - скор 0.9396

In [14]:
all_confident.shape

(13390, 25)

In [15]:
all_review.shape

(122, 25)

In [ ]:
print("=== ПОДГОТОВКА САБМИТА (WORKER INDICES) ===")

# 1. Объединяем результаты
submission_source = pd.concat([all_confident, all_review], ignore_index=True)

print(f"Всего найденных пар (Confident + Review): {len(submission_source)}")

# 2. Извлекаем индексы СОТРУДНИКОВ (emp_idx)
suspect_employees = submission_source['emp_idx'].dropna().unique()

# 3. Приводим к формату uint64 и сортируем
suspect_employees = np.sort(suspect_employees).astype(np.uint64)

print(f"Уникальных сотрудников под подозрением: {len(suspect_employees)}")

# 4. Создаем DataFrame нужного формата
# id (от 0 до N), orig_index (индекс из employees)
res = pd.DataFrame({
    'orig_index': suspect_employees
}).reset_index(names='id')

# Проверка
print("\nПре-вью файла:")
print(res.head())
print(res.dtypes)

# 5. Сохраняем
res.to_parquet('submission.parquet', index=False)
print("\n✅ Файл submission.parquet готов к отправке!")

=== ПОДГОТОВКА САБМИТА (WORKER INDICES) ===
Всего найденных пар (Confident + Review): 13512
Уникальных сотрудников под подозрением: 13462

Пре-вью файла:
   id  orig_index
0   0         515
1   1         583
2   2         588
3   3         677
4   4        1012
id             int64
orig_index    uint64
dtype: object

✅ Файл submission.parquet готов к отправке!


In [20]:
all_review.head(100)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,orc_idx,inn,emp_idx,fio_orc,fio_emp,gender_orc,gender_emp,passport_orc,passport_emp,inn_orc,inn_emp,birthdate_orc,birthdate_emp,score_fio,score_gender,score_pass,score_inn,score_date,total_score,is_confident,passport,birthdate,key_sur,key_nf,tfidf_sim
1110,14075,579171383322,428185,калиничев* хартя валерьевна,ка*еничева мария валериевна,ж,ж,3473673366,2472672266,579171383322,579171383322,1988-03-02,,1,1,-1,3,0,4,False,<NA>,<NA>,NaN,NaN,NaN
134,16422,<NA>,1422611,мошная ульяна леговна,ошная емельяна леваевн*,ж,ж,3219880449,,840737303004,840733703004,1960-06-05,1960-06-05,-1,1,0,2,2,4,False,<NA>,<NA>,NaN,NaN,NaN
1250,9674,<NA>,1238892,орлов дмитрий алексеевич,залов дмитрий олексеевич,м,м,4378863614,4378863614,,732544853065,1987-11-29,1985-11-29,-1,1,3,0,1,4,False,4378863614,<NA>,NaN,NaN,NaN
1430,11058,<NA>,937978,барышнкова оська анатолиевна,ажышникова ольга онатольевна,ж,ж,1761177181,1761177181,,395814532662,1998-04-24,1998-04-42,-1,1,3,0,1,4,False,1761177181,<NA>,NaN,NaN,NaN
1561,12142,<NA>,1905328,бордиловский надилхан елкаму,хърдиловский на*илхан мелкаму,м,м,3563308966,3563308966,,152973980028,1974-10-12,1974-10-21,-1,1,3,0,1,4,False,3563308966,<NA>,NaN,NaN,NaN
2579,20237,<NA>,1359286,боа*чкина веа* евановна,огочкина вера ивановна,ж,ж,8186083816,8186083816,746390779377,846390789387,1981-08-12,1981-08-12,-1,1,3,-1,2,4,False,8186083816,<NA>,NaN,NaN,NaN
2906,22870,<NA>,1413863,лолотилова айиша арешаровна,золотелова ойеша ришаорвна,ж,ж,5809103018,5809103018,,690661576196,1983-03-28,1983-03-25,-1,1,3,0,1,4,False,5809103018,<NA>,NaN,NaN,NaN
3408,26472,<NA>,1453561,костенко виталий онатольевич,ксточко виталий анатольевич,м,м,5712834151,5712834151,,394223545674,1993-04-06,1939-04-06,-1,1,3,0,1,4,False,5712834151,<NA>,NaN,NaN,NaN
5237,41136,<NA>,1159752,торбпна алина георгиевна,торбино лида георгиевна,ж,ж,2508379571,2508379571,224835449446,221835119116,,1990-03-25,1,1,3,-1,0,4,False,2508379571,<NA>,NaN,NaN,NaN
5931,46638,<NA>,747128,бочарова галина николаевна,гончарова галина николаевна,ж,ж,4899576890,4899576890,630218216606,503218215605,,,1,1,3,-1,0,4,False,4899576890,<NA>,NaN,NaN,NaN


In [21]:
all_confident[all_confident["total_score"] == 3].head(100)

,orc_idx,inn,emp_idx,fio_orc,fio_emp,gender_orc,gender_emp,passport_orc,passport_emp,inn_orc,inn_emp,birthdate_orc,birthdate_emp,score_fio,score_gender,score_pass,score_inn,score_date,total_score,is_confident,passport,birthdate,key_sur,key_nf,tfidf_sim
